# Réplica de Aradillas (2018) — ENIGH 2014
## Poder de mercado y bienestar social en hogares mexicanos

Réplica en Python del estudio de **Aradillas López, A. (2018)**, *"Estudio sobre el impacto
que tiene el poder de mercado en el bienestar de los hogares mexicanos"* (COFECE), y base
para su actualización a ENIGH 2022 y 2024.

Este notebook **solo orquesta**: el cálculo vive en `aradillas_core.py` y la carga de datos
en `datos_2014.py`. Ejecutar desde el directorio que contiene `Replica_COFECE/`.

---

### Fuente de verdad: el código Gauss, no el paper

`CD/programa_ENIGH_2014.g` es el programa original, obtenido por solicitud de acceso a la
información. **Ante cualquier discrepancia manda el Gauss.** Esta decisión es deliberada:
al replicar se encontraron cuatro casos donde corregir un error verificado *aleja* los
resultados de las cifras publicadas, lo que sugiere que el paper incorpora errores que se
compensan entre sí.

Dos hallazgos estructurales del Gauss que el paper no refleja:

* El paper describe una estimación en **dos etapas (OLS + GMM)**; el Gauss implementa
  **solo el bucle OLS** de 16 iteraciones (l.2452–5753). Los parámetros publicados son OLS.
* El Gauss aplica un **filtro de vivienda propia** (l.1101) que descarta 5,215 hogares
  (27 %) y que **el paper nunca menciona**. Se conserva por fidelidad al Gauss.

### Correcciones incorporadas

Cada una verificada contra el Gauss, con su ubicación en el código:

| # | qué estaba mal | dónde |
|---|---|---|
| N4 | El archivo de precios de referencia se leía como si sus columnas fueran contiguas; el Gauss salta las columnas 62, 65 y 66. Seis productos desplazados: transporte aéreo tomaba 138 pesos en vez de 2,279 | `datos_2014.REF_COL_GAUSS` |
| N6 | Se descartaban los hogares cuya demanda contrafactual subía antes de agregar; el Gauss agrega sobre todos (l.6146) | `elasticidades()` |
| **N8** | **Causa de la compresión de elasticidades.** Los pesos del índice Divisia no sumaban 1: el piso EPS se aplicaba al total en vez de a cada producto, así que para hogares sin gasto real `w_j = 1` en cada producto y el índice se volvía el *producto* de los precios (std ln P = 13.8 en Frutas) | `divisia_price_index()` |
| N8b | La misma causa en el gasto de categoría, que alimenta las participaciones | `datos_2014.cargar()` |
| N10 | El Gini usaba `ing_total` sobre la muestra filtrada; el Gauss usa `ing_mon` sobre los 19,124 hogares completos, con contrafactual multiplicativo por decil (l.938, 7410–7433) | `gini()` |
| N11 | El denominador de la VE debe ser `ing_mon`, no `ing_total` | `cuadro_10()` |
| N12 | `.mean()` en vez de `np.nanmean()`: con `ing_mon` hay hogares con ingreso monetario cero | `cuadro_10()` |
| N13 | El *fallback* de la utilidad contrafactual resolvía a precios **originales**, dejando a esos hogares sin respuesta al cambio de precio | `elasticidades()` |
| N14 | Newton+`minimize_scalar` no resolvía la ecuación en el 42 % de los hogares (error p90 = 2.21); se usan las raíces exactas del cúbico | `ModeloEASI.resolver_utilidad()` |

### Qué replica y qué no

| resultado | réplica | paper |
|---|---|---|
| **Gini observado** | **0.481** | **0.481** |
| Gini contrafactual | 0.451 | 0.446 |
| Cuadro 5 (regional) | 8/8 dentro de ±0.15 | |
| Cuadro 8 (β_η) | dentro del rango publicado (0.02–1.48) | |
| Cuadro 4 (elasticidades) | MAE 0.239, 10/13 dentro de ±0.30 | |

**Limitaciones conocidas.** La muestra final es de 8,940 hogares contra los 15,586 que
reporta el paper, y ninguna combinación de filtros reproduce esa cifra. `Carne res` y
`Carnes procesadas` son las categorías con mayor desviación: en ambas, ~64 % de los hogares
no registra compra, así que sus índices de precio dependen del reparto equitativo que impone
el piso EPS —un artefacto que el Gauss también tiene—. En res, las vísceras pesan 22.8 % en
el índice cuando su participación real es 3.4 %.


In [1]:
import sys
sys.path.insert(0, "Replica_COFECE/Codigo")
import numpy as np

import datos_2014
from aradillas_core import (estimar_easi, reconstruir_matrices, ModeloEASI,
                            demandas_marshallianas, elasticidades,
                            estimar_markups, variacion_equivalente,
                            cuadro_10, gini, sectores_significativos)

DATA_DIR = "Replica_COFECE/Data_2014/"
print("Módulos cargados.")

Módulos cargados.


## 1–2. Precios locales y microdatos ENIGH 2014

`datos_2014.cargar()` hace todo: deflacta los precios de junio 2011 con INPC (INPP para
materiales) a la ventana de levantamiento ago–nov 2014, aplica los filtros de muestra del
Gauss (l.1101), asigna a cada hogar la ciudad INPC más cercana con corte a 400 km, agrega
en 12 categorías y construye los índices de precio Divisia.

Cascada de muestra esperada: 19,124 → 12,592 (filtros) → 12,461 (400 km) → 12,372
(al menos una categoría con gasto ≥ 10).

In [2]:
datos = datos_2014.cargar(DATA_DIR)
precios_46_estado = np.loadtxt(
    DATA_DIR + "precios_promedio_46_ciudades_junio_2011.asc")[:, 0]

Hogares después de filtros básicos: 12592
Hogares después de filtro de distancia (<=400 km): 12461


Hogares finales en la muestra: 12372
ENIGH 2014: 12372 hogares, 12 categorías, 46 ciudades


## 4. Sistema aproximado de demanda EASI

16 iteraciones de OLS con simetría de **B** y **Aℓ** impuesta, recortando el 1 % de cada
cola en cada paso. El recorte es acumulativo y se lleva el 28 % de la muestra
(12,372 → 8,940): es lo que hace el Gauss, aunque el paper no lo menciona.

Nota: el criterio de convergencia **no converge** y oscila, porque cada iteración estima
sobre una muestra distinta. Es inherente al algoritmo del Gauss, no un error de la réplica.

In [3]:
print("Estimando sistema EASI (16 iteraciones, trim=1%)...")
res = estimar_easi(datos.precios_ln, datos.w, datos.gasto_total, datos.Z,
                   n_cat=datos.n_cat)
datos = datos.submuestra(res["mask"])
print(datos.resumen())

Estimando sistema EASI (16 iteraciones, trim=1%)...


  Iteración  1: (primera estimación)  N=12372


  Iteración  2: criterio = 1.610090  N=12124


  Iteración  3: criterio = 0.457116  N=11880


  Iteración  4: criterio = 0.232612  N=11642


...

  Iteración 12: criterio = 0.096302  N=9896


  Iteración 13: criterio = 0.045224  N=9698


  Iteración 14: criterio = 0.094726  N=9504


  Iteración 15: criterio = 0.069669  N=9312


  Iteración 16: criterio = 0.064200  N=9124

Estimación completada. Muestra final: 8940 hogares.
ENIGH 2014: 8940 hogares, 12 categorías, 46 ciudades


## 5. Matrices de parámetros y utilidad indirecta exacta

La ecuación de costo `C(p,u,z,ε) = ln x` es un **polinomio cúbico en u**, así que tiene
solución cerrada. Se intenta Newton amortiguado primero por velocidad y se cae a las raíces
exactas cuando falla (corrección N14). El Gauss resuelve lo mismo con `optmum()`, que hace
line search desde la utilidad aproximada `u0`; por eso se elige la raíz real más cercana
a `u0`.

In [4]:
mats = reconstruir_matrices(res["beta"], n_cat=datos.n_cat)
modelo = ModeloEASI(**mats)
print(f"B simétrica: {np.allclose(mats['B_mat'], mats['B_mat'].T)}   "
      f"|Σ cols B| max: {np.abs(mats['B_mat'].sum(0)).max():.2e}")

epsilon = res["epsilon"]
util = modelo.utilidad_indirecta(datos.precios_ln, datos.Z, epsilon,
                                 datos.w, datos.gasto_total)

B simétrica: True   |Σ cols B| max: 5.55e-17
  0/8940  (fallbacks: 0)


  2000/8940  (fallbacks: 49)
  4000/8940  (fallbacks: 111)


  6000/8940  (fallbacks: 182)
  8000/8940  (fallbacks: 232)



Newton converge: 8681/8940 (97.1%)
Utilidad: media=1.1277  std=1.3095  rango [-1.814, 14.849]


## 6. Demandas Marshallianas y elasticidades

Las demandas agregadas se ponderan por el factor de expansión π_h de la ENIGH. Autobús
foráneo y transporte aéreo comparten la categoría 11: sus elasticidades se obtienen
perturbando el sub-precio que a cada uno le corresponde dentro del índice Divisia de la
categoría, replicando `factor_contrafactual_aereo/_foraneo` del Gauss (l.5926–5929).

In [5]:
demandas, _ = demandas_marshallianas(
    modelo, datos.precios_ln, datos.Z, util, epsilon,
    datos.gasto_total, datos.factor_expansion, datos.n_cat)

factor_cf = 1.25
g_a, g_e = datos.subcat["g_autobus"], datos.subcat["g_aereo"]
p_a, p_e = datos.subcat["p_autobus"], datos.subcat["p_aereo"]
w_a = g_a / (g_a + g_e); w_e = g_e / (g_a + g_e)
pi = datos.factor_expansion
wba = (w_a*pi).sum()/pi.sum(); wbe = (w_e*pi).sum()/pi.sum()
k_t = (wba**(-wba)) * (wbe**(-wbe))

def cf_trans(sub):
    po = (1/k_t)*((p_a/w_a)**w_a)*((p_e/w_e)**w_e)
    if sub == "aereo":
        pc = (1/k_t)*((p_a/w_a)**w_a)*(((p_e*factor_cf)/w_e)**w_e)
    else:
        pc = (1/k_t)*(((p_a*factor_cf)/w_a)**w_a)*((p_e/w_e)**w_e)
    return pc/po

cats_n = datos.nombres_cat + ["Trans. aéreo", "Autobús foráneo"]
print("Calculando elasticidades...")
elastic_nac, elastic_46 = elasticidades(
    modelo, datos.precios_ln, datos.Z, epsilon, datos.w, datos.gasto_total,
    pi, demandas, datos.ciudad, datos.n_ciudades, datos.n_cat, util,
    factor_cf=factor_cf,
    factores_extra=[("Trans. aéreo", 10, cf_trans("aereo")),
                    ("Autobús foráneo", 10, cf_trans("foraneo"))],
    nombres=datos.nombres_cat)

Calculando elasticidades...
  [ 1] Tortillas              

e=0.842
  [ 2] Pan                    

e=1.737
  [ 3] Pollo+Huevo            

e=1.065
  [ 4] Carne res              

e=1.519
  [ 5] Carnes proc.           

...

e=0.690
  [11] Transporte foráneo     

e=0.800
  [12] Materiales             

e=1.014
  [13] Trans. aéreo           

e=1.101
  [14] Autobús foráneo        

e=0.436


In [6]:

# ---------------------------------------------------------------
# 6.3  Cuadros 4 y 5 — comparación final
# ---------------------------------------------------------------
paper_e = {
    'Tortillas':1.054,'Pan':1.462,'Pollo+Huevo':1.261,'Carne res':0.735,
    'Carnes proc.':0.968,'Lácteos':1.289,'Frutas':1.415,'Verduras':1.389,
    'Bebidas':1.110,'Medicamentos':0.943,'Materiales':0.934,
    'Trans. aéreo':1.246,'Autobús foráneo':0.847
}
idx_map = [0,1,2,3,4,5,6,7,8,9,11,12,13]

print("=== CUADRO 4: ELASTICIDADES NACIONALES ===")
print(f"{'Categoría':<22} {'Nuestra':>8} {'Paper':>8} {'Dif':>8}")
print("-"*54)
difs=[]
for cat, pfp in zip(paper_e, idx_map):
    n=abs(elastic_nac[pfp]); p=paper_e[cat]; d=n-p; difs.append(abs(d))
    flag = "✓" if abs(d)<0.15 else ("~" if abs(d)<0.30 else "⚠")
    print(f"  {cat:<20} {n:>8.3f} {p:>8.3f} {d:>8.3f} {flag}")
mae=sum(difs)/len(difs)
print(f"\nMAE: {mae:.3f}")
print(f"✓ ±0.15: {sum(1 for d in difs if d<0.15)}/13")
print(f"~ ±0.30: {sum(1 for d in difs if d<0.30)}/13")

REGIONES={'Noroeste':[2,3,8,10,25,26],'Noreste':[5,19,28],
          'Oeste':[6,14,16,18],'Este':[13,21,29,30],
          'Centro Norte':[1,11,22,24,32],'Centro Sur':[9,15,17],
          'Suroeste':[7,12,20],'Sureste':[4,23,27,31]}
cr={i:reg for reg,es in REGIONES.items()
    for i in range(46) if int(precios_46_estado[i]) in es}
ref_r={'Noroeste':1.232,'Noreste':1.171,'Oeste':1.240,'Este':1.237,
       'Centro Norte':1.209,'Centro Sur':1.168,'Suroeste':1.179,'Sureste':1.165}

print(f"\n=== CUADRO 5: ALIMENTOS Y BEBIDAS POR REGIÓN ===")
print(f"{'Región':<14} {'Nuestra':>8} {'Paper':>8} {'Dif':>6}")
print("-"*42)
for reg,pref in ref_r.items():
    cs=[c for c,r in cr.items() if r==reg]
    vs=[abs(elastic_46[c,pfp]) for c in cs for pfp in range(9)
        if abs(elastic_46[c,pfp])>0]
    e=np.mean(vs) if vs else 0.
    flag="✓" if abs(e-pref)<0.15 else ("~" if abs(e-pref)<0.30 else "⚠")
    print(f"  {reg:<12} {e:>8.3f} {pref:>8.3f} {e-pref:>6.3f} {flag}")


=== CUADRO 4: ELASTICIDADES NACIONALES ===
Categoría               Nuestra    Paper      Dif
------------------------------------------------------
  Tortillas               0.842    1.054   -0.212 ~
  Pan                     1.737    1.462    0.275 ~
  Pollo+Huevo             1.065    1.261   -0.196 ~
  Carne res               1.519    0.735    0.784 ⚠
  Carnes proc.            1.315    0.968    0.347 ⚠
  Lácteos                 1.397    1.289    0.108 ✓
  Frutas                  1.400    1.415   -0.015 ✓
  Verduras                1.280    1.389   -0.109 ✓
  Bebidas                 0.935    1.110   -0.175 ~
  Medicamentos            0.690    0.943   -0.253 ~
  Materiales              1.014    0.934    0.080 ✓
  Trans. aéreo            1.101    1.246   -0.145 ✓
  Autobús foráneo         0.436    0.847   -0.411 ⚠

MAE: 0.239
✓ ±0.15: 5/13
~ ±0.30: 10/13

=== CUADRO 5: ALIMENTOS Y BEBIDAS POR REGIÓN ===
Región          Nuestra    Paper    Dif
------------------------------------------
  

## 7. Markups y poder de mercado (NEIO)

Modelo de sobreprecios de Bresnahan (1989): el precio de cada ciudad se regresa sobre
η = −p/ε y las variables de costo de los Censos Económicos, con errores estándar de White.

**Cómo leer el Cuadro 8.** Si las elasticidades estuvieran comprimidas hacia −1, entonces
η ≈ p y la regresión devolvería β ≈ 1 con estadísticos t enormes por colinealidad casi
perfecta — una identidad algebraica, no poder de mercado. Que los β caigan en el rango
publicado (0.02–1.48) con t de 4–35, y que algún sector salga **no significativo**, es la
señal de que la estimación es sana.

In [7]:
# Precios por ciudad: se ponderan con las participaciones de la muestra
# VIGENTE (post-trim), como en el Gauss. Se añaden las dos sub-categorías de
# transporte para que las columnas casen con elastic_46.
comp = dict(datos.composicion)
comp["Trans. aéreo"] = ["transporte_aereo"]
comp["Autobús foráneo"] = ["autobus_foraneo"]
P_cat_46 = datos.precios_por_ciudad(comp)

mk = estimar_markups(P_cat_46, elastic_46, datos.vars_costos)
beta_eta, t_stat_eta, markup_46 = mk["beta_eta"], mk["t_eta"], mk["markup"]

for j, nom in enumerate(cats_n):
    sig = "***" if abs(t_stat_eta[j]) >= 2.326 else (
          "**" if abs(t_stat_eta[j]) >= 1.645 else "  ")
    print(f"  [{j+1:2d}] {nom:<22} β={beta_eta[j]:>7.3f}  "
          f"t={t_stat_eta[j]:>7.3f} {sig}")

  [ 1] Tortillas              β=  0.146  t=  2.581 ***
  [ 2] Pan                    β=  0.646  t=  2.830 ***
  [ 3] Pollo+Huevo            β=  0.283  t=  3.212 ***
  [ 4] Carne res              β=  0.375  t=  3.727 ***
  [ 5] Carnes proc.           β=  0.001  t=  0.008   
  [ 6] Lácteos                β=  0.578  t=  7.204 ***
  [ 7] Frutas                 β=  0.801  t=  6.414 ***
  [ 8] Verduras               β=  0.538  t=  5.801 ***
  [ 9] Bebidas                β=  0.189  t=  2.537 ***
  [10] Medicamentos           β=  0.013  t=  0.679   
  [11] Transporte foráneo     β=  0.011  t=  0.318   
  [12] Materiales             β=  0.084  t=  2.473 ***
  [13] Trans. aéreo           β=  0.051  t=  0.307   
  [14] Autobús foráneo        β= -0.007  t= -0.212   


In [8]:
print("\n=== CUADRO 8: PARÁMETROS DE PODER DE MERCADO β_η ===")
paper_b   = {'Tortillas':0.183,'Pan':1.477,'Pollo+Huevo':0.139,'Carne res':0.047,
             'Carnes proc.':0.017,'Lácteos':0.626,'Frutas':1.120,'Verduras':0.328,
             'Bebidas':0.047,'Medicamentos':0.026,'Materiales':0.493,
             'Trans. aéreo':0.196,'Autobús foráneo':0.081}
paper_t_v = {'Tortillas':3.223,'Pan':16.268,'Pollo+Huevo':1.796,'Carne res':2.851,
             'Carnes proc.':0.906,'Lácteos':3.933,'Frutas':12.033,'Verduras':3.249,
             'Bebidas':1.531,'Medicamentos':1.566,'Materiales':5.535,
             'Trans. aéreo':4.368,'Autobús foráneo':2.718}
idx8 = [0,1,2,3,4,5,6,7,8,9,11,12,13]
print(f"{'Categoría':<22} {'β nuestro':>10} {'β paper':>8} {'t nuestro':>10} {'t paper':>8}")
print("-"*62)
for cat,pfp in zip(paper_b.keys(),idx8):
    b=beta_eta[pfp]; pb=paper_b[cat]; t=t_stat_eta[pfp]; pt=paper_t_v[cat]
    sig="***" if abs(t)>=2.326 else ("**" if abs(t)>=1.645 else "  ")
    print(f"  {cat:<20} {b:>10.3f} {pb:>8.3f} {t:>10.3f} {pt:>8.3f} {sig}")

# Cuadro 9
paper_sp = {'Frutas':238.52,'Pan':199.95,'Materiales':113.25,'Lácteos':95.43,
            'Verduras':30.47,'Trans. aéreo':27.40,'Tortillas':26.19,
            'Autobús foráneo':14.54,'Carne res':8.13,'Pollo+Huevo':14.02,
            'Bebidas':4.85,'Medicamentos':4.36,'Carnes proc.':1.86}
print("\n=== CUADRO 9: SOBREPRECIOS ===")
print(f"{'Categoría':<22} {'Nuestro (%)':>12} {'Paper (%)':>10}")
print("-"*48)
sp_list = []
for cat, pfp in zip(paper_b.keys(), idx8):
    mv = markup_46[:,pfp][markup_46[:,pfp]>1]
    sp = (mv.mean()-1)*100 if len(mv)>0 else 0.
    sp_list.append(sp)
    print(f"  {cat:<20} {sp:>12.2f} {paper_sp.get(cat,0):>10.2f}")
print(f"\n  Promedio: {np.mean(sp_list):.2f}%  (Paper: 98.23%)")



=== CUADRO 8: PARÁMETROS DE PODER DE MERCADO β_η ===
Categoría               β nuestro  β paper  t nuestro  t paper
--------------------------------------------------------------
  Tortillas                 0.146    0.183      2.581    3.223 ***
  Pan                       0.646    1.477      2.830   16.268 ***
  Pollo+Huevo               0.283    0.139      3.212    1.796 ***
  Carne res                 0.375    0.047      3.727    2.851 ***
  Carnes proc.              0.001    0.017      0.008    0.906   
  Lácteos                   0.578    0.626      7.204    3.933 ***
  Frutas                    0.801    1.120      6.414   12.033 ***
  Verduras                  0.538    0.328      5.801    3.249 ***
  Bebidas                   0.189    0.047      2.537    1.531 ***
  Medicamentos              0.013    0.026      0.679    1.566   
  Materiales                0.084    0.493      2.473    5.535 ***
  Trans. aéreo              0.051    0.196      0.307    4.368   
  Autobús foráneo  

## 8. Variación equivalente y bienestar

El contrafactual elimina el sobreprecio en los sectores con β_η significativo. El Gini se
calcula como en el Gauss (l.938, 7410–7433): sobre `ing_mon` de la **muestra completa** del
concentrado —la l.938 va antes del filtro de la l.1101, así que no aplica filtros ni
recorte— y con ajuste **multiplicativo por decil**, `ingreso × 1/(1 − tasa)`, no sumando la
VE en pesos.

In [9]:
sig_95 = sectores_significativos(t_stat_eta[:datos.n_cat],
                                 beta_eta[:datos.n_cat])   # V1
print("Sectores significativos (V1: t >= 2.326, 99%):")
for cat, s in zip(datos.nombres_cat, sig_95):
    print(f"  {cat:<22} {'✓' if s else '✗'}")

mk_hogar = mk["markup_lerner"][datos.ciudad][:, :datos.n_cat]  # V2
print("\nCalculando VE...")
VE = variacion_equivalente(modelo, datos.precios_ln, datos.Z, epsilon,
                           datos.w, datos.gasto_total, mk_hogar, sig_95)
print(f"\nVE media:   ${VE.mean():.0f}  (paper: $1,497)")
print(f"VE mediana: ${np.median(VE[VE>0]):.0f}")

c10 = cuadro_10(VE, datos.ingreso_total)  # V3: el Gauss divide por ing_total (col 22, l.6519)
print(f"VE/ingreso: {c10['total']['pct']:.1f}%  (paper: 15.7%)")

Sectores significativos (V1: t >= 2.326, 99%):
  Tortillas              ✓
  Pan                    ✓
  Pollo+Huevo            ✓
  Carne res              ✓
  Carnes proc.           ✗
  Lácteos                ✓
  Frutas                 ✓
  Verduras               ✓
  Bebidas                ✓
  Medicamentos           ✗
  Transporte foráneo     ✗
  Materiales             ✓

Calculando VE...
  0/8940...


  2000/8940...
  4000/8940...


  6000/8940...
  8000/8940...



VE media:   $4495  (paper: $1,497)
VE mediana: $3736
VE/ingreso: 13.0%  (paper: 15.7%)


In [10]:
paper_m = [841,1097,1286,1410,1487,1613,1738,1907,2052,2237,1497]
paper_p = [30.9,23.6,21.4,18.9,16.7,15.1,13.6,11.9,9.5,5.7,15.7]

print("=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===")
print(f"{'Decil':<6} {'VE($)':>8} {'Paper':>7} {'VE/Ing%':>9} {'Paper%':>8}")
print("-"*44)
for f in c10["deciles"]:
    d = f["decil"]
    print(f"  {d:<4} {f['VE']:>8.0f} {paper_m[d-1]:>7} "
          f"{f['pct']:>9.1f} {paper_p[d-1]:>8.1f}")
print(f"  {'Tot':<4} {c10['total']['VE']:>8.0f} {paper_m[-1]:>7} "
      f"{c10['total']['pct']:>9.1f} {paper_p[-1]:>8.1f}")
print(f"\nRegresividad: {c10['regresividad']:.2f}x  (paper: 4.42x)")

g = gini(datos.ingreso_mon_completo, c10["tasas"])
print(f"\nGini observado:     {g['observado']:.3f}  (paper: 0.481)   "
      f"[ing_mon, N={g['n']}]")
print(f"Gini contrafactual: {g['contrafactual']:.3f}  (paper: 0.446)")
print(f"Reducción:          {g['reduccion_pct']:.1f}%  (paper: 7.3%)")

=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===
Decil     VE($)   Paper   VE/Ing%   Paper%
--------------------------------------------
  1        2056     841      25.8     30.9
  2        2770    1097      20.0     23.6
  3        3266    1286      19.0     21.4
  4        3473    1410      15.8     18.9
  5        3716    1487      14.1     16.7
  6        4277    1613      13.7     15.1
  7        4450    1738      11.6     13.6
  8        4676    1907       9.7     11.9
  9        4786    2052       7.5      9.5
  10       5566    2237       4.5      5.7
  Tot      3736    1497      13.0     15.7

Regresividad: 5.68x  (paper: 4.42x)

Gini observado:     0.481  (paper: 0.481)   [ing_mon, N=19091]
Gini contrafactual: 0.454  (paper: 0.446)
Reducción:          5.7%  (paper: 7.3%)
